# 08 — Architecture visualization

Pass the same VisDrone image through each architecture. Print actual module names first, then attach hooks to selected real modules. This avoids assuming undocumented VMamba or framework internals.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/visdrone_architecture_benchmark"
GITHUB_USERNAME = "Harryphan72007"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/aerial-object-detection-benchmark.git"
REPO_DIR = "/content/aerial-object-detection-benchmark"
!test -d {REPO_DIR}/.git || git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements-colab.txt
!pip install -q -e .

In [ ]:
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
collect_environment()

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def list_modules(model, contains=None):
    for name, module in model.named_modules():
        if not contains or contains.lower() in name.lower(): print(name, module.__class__.__name__)

def capture_modules(model, names):
    outputs={}; hooks=[]
    for name,module in model.named_modules():
        if name in names:
            hooks.append(module.register_forward_hook(lambda m,i,o,n=name: outputs.__setitem__(n,o)))
    return outputs,hooks

def activation_views(tensor):
    if isinstance(tensor,(list,tuple)): tensor=tensor[0]
    if hasattr(tensor,"last_hidden_state"): tensor=tensor.last_hidden_state
    x=tensor.detach().float().cpu()
    if x.ndim==4:
        mean=x.mean(1)[0]; maximum=x.max(1).values[0]
        c,h,w=x.shape[1:]; flat=x[0].reshape(c,-1).T.numpy(); pca=PCA(1).fit_transform(flat).reshape(h,w)
        return mean.numpy(),maximum.numpy(),pca
    return None

## Load a run and inspect names

For Faster R-CNN inspect backbone stages, neck/FPN, RPN, and RoI modules. For Swin inspect patch embedding and stage blocks. For VMamba search for `backbone`, `layers`, `blocks`, `ss2d`, or `op` only after seeing the installed names. For RT-DETR inspect backbone, encoder, decoder, query embeddings, and intermediate hidden states exposed by documented outputs.

In [ ]:
from src.training.checkpointing import RunRegistry
from src.models.registry import create_adapter
from src.utils.serialization import read_yaml
registry=RunRegistry(paths)
RUN_ID=None  # choose a completed run
if RUN_ID:
    run=next(r for r in registry.list_available_runs(status=None) if r["run_id"]==RUN_ID)
    run_dir=paths.run_dir(run["model_id"],RUN_ID); cfg=read_yaml(run_dir/"model_config.yaml")
    if run["framework"] in {"mmdetection","vmamba_mmdetection"}: cfg["resolved_framework_config"]=str(run_dir/"runtime_config.py")
    adapter=create_adapter(run["model_id"]); model=adapter.load_model(registry.load_checkpoint_from_registry(RUN_ID),cfg)
    list_modules(model)

## Effective receptive field

Backpropagate from a selected central activation to input pixels, normalize absolute gradients, and compare spread. Keep preprocessing and selected semantic level consistent across models.

## Same-image four-architecture comparison

This cell selects the best completed two-class run for each primary model, uses the same image, renders predictions side by side, and captures only real module names from the installed implementation.


In [ ]:
from src.data.dataloaders import CocoDetectionRecords
from src.evaluation.visualization import select_module_names, capture_module_outputs, plot_activation_views, draw_predictions
from IPython.display import display
primary_models = ["faster_rcnn_resnet50", "faster_rcnn_swin_t", "faster_rcnn_vmamba_t", "rtdetrv2_l"]
records = CocoDetectionRecords(paths.coco("2class")/"val", paths.coco("2class")/"annotations/instances_val.json")
shared_image = records[0]["image"]
keyword_map = {
    "faster_rcnn_resnet50": ["backbone.layer", "neck", "rpn_head", "roi_head"],
    "faster_rcnn_swin_t": ["patch_embed", "backbone.stages", "neck", "rpn_head", "roi_head"],
    "faster_rcnn_vmamba_t": ["patch_embed", "backbone.layers", "ss2d", "op", "neck"],
    "rtdetrv2_l": ["backbone", "encoder", "decoder", "query"],
}
for model_id in primary_models:
    candidates = registry.list_available_runs(model_id, "2class")
    if not candidates:
        print(model_id, "has no completed run")
        continue
    run = max(candidates, key=lambda item: float(item.get("best_validation_map", 0)))
    run_dir = paths.run_dir(model_id, run["run_id"])
    cfg = read_yaml(run_dir/"model_config.yaml")
    cfg["input_resolution"] = run["input_resolution"]
    if run["framework"] in {"mmdetection", "vmamba_mmdetection"}:
        cfg["resolved_framework_config"] = str(run_dir/"runtime_config.py")
    adapter = create_adapter(model_id)
    model = adapter.load_model(registry.load_checkpoint_from_registry(run["run_id"]), cfg)
    names = select_module_names(model, keyword_map[model_id], limit=16)
    outputs, handles = capture_module_outputs(model, names)
    prediction = adapter.predict([shared_image])[0]
    for handle in handles: handle.remove()
    print(model_id, "stage modules:", names)
    display(draw_predictions(shared_image, prediction, run["class_names"], threshold=0.25))
    try:
        display(plot_activation_views(outputs, maximum_modules=5))
    except RuntimeError as error:
        print(error)
